<a href="https://colab.research.google.com/github/fergogu27-ctrl/EDPII/blob/main/Actividad5_uniformizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Actividad 5 - Método de Uniformización para una CMTC

- Ejercicio 3: Cálculo de $P(0.5)$, $P(1)$ y $P(5)$ usando la aproximación
$M \approx \max\{rt + 5\sqrt{rt},20\}.$
- Verificación de Chapman-Kolmogorov:
$P(1) \approx P(0.5)P(0.5).$
- Ejercicio 4.2: Algoritmo de uniformización con tolerancia $\varepsilon=0.00001.$

La matriz de tasas usada es:
$$R =
\begin{pmatrix}
0 & 2 & 3 & 0\\
4 & 0 & 2 & 0\\
0 & 2 & 0 & 2\\
1 & 0 & 3 & 0
\end{pmatrix}.$$


In [1]:
import numpy as np
import math
import pandas as pd

In [2]:
np.set_printoptions(precision=6, suppress=True)

R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

def matriz_uniformizada(R, r=None):
    tasas_salida = R.sum(axis=1)

    if r is None:
        r = tasas_salida.max()

    P_hat = R / r
    np.fill_diagonal(P_hat, 1 - tasas_salida / r)

    return P_hat, r, tasas_salida

P_hat, r, tasas_salida = matriz_uniformizada(R)

print("Tasas de salida r_i:")
print(tasas_salida)

print("\nTasa de uniformización r:")
print(r)

print("\nMatriz P_hat:")
print(P_hat)

print("\nSuma de los renglones de P_hat:")
print(P_hat.sum(axis=1))


Tasas de salida r_i:
[5. 6. 4. 4.]

Tasa de uniformización r:
6.0

Matriz P_hat:
[[0.166667 0.333333 0.5      0.      ]
 [0.666667 0.       0.333333 0.      ]
 [0.       0.333333 0.333333 0.333333]
 [0.166667 0.       0.5      0.333333]]

Suma de los renglones de P_hat:
[1. 1. 1. 1.]


In [5]:
def uniformizacion_con_M(R, t, M=None):
    P_hat, r, _ = matriz_uniformizada(R)
    n = R.shape[0]
    lam = r * t

    if M is None:
        M = math.ceil(max(lam + 5 * math.sqrt(lam), 20))

    B = np.zeros((n, n))
    A = np.eye(n)
    c = math.exp(-lam)

    for k in range(M + 1):
        if k > 0:
            A = A @ P_hat
            c = c * lam / k
        B = B + c * A

    return B, M


def uniformizacion_con_tolerancia(R, t, eps=1e-5):
    P_hat, r, _ = matriz_uniformizada(R)
    n = R.shape[0]
    lam = r * t

    A = P_hat.copy()
    c = math.exp(-lam)
    B = c * np.eye(n)
    suma = c
    k = 1

    while suma < 1 - eps:
        c = c * lam / k
        B = B + c * A
        A = A @ P_hat
        suma = suma + c
        k = k + 1

    M = k - 1
    cola = 1 - suma

    return B, M, cola


def tabla_matriz(M, nombre="Matriz"):
    return pd.DataFrame(
        np.round(M, 6),
        index=[f"estado {i}" for i in range(1, M.shape[0] + 1)],
        columns=[f"estado {j}" for j in range(1, M.shape[1] + 1)]
    )


In [6]:
# Ejercicio 3: cálculo de P(0.5), P(1) y P(5)

tiempos = [0.5, 1, 5]
resultados_M = {}

for t in tiempos:
    P_t, M = uniformizacion_con_M(R, t)
    resultados_M[t] = P_t

    print(f"\nt = {t}")
    print(f"M usado = {M}")
    print(P_t)
    print("Suma de renglones:", P_t.sum(axis=1))



t = 0.5
M usado = 20
[[0.250609 0.216965 0.386657 0.14577 ]
 [0.253135 0.238361 0.374409 0.134095]
 [0.169119 0.193615 0.420301 0.216965]
 [0.158017 0.157445 0.398332 0.286206]]
Suma de renglones: [1. 1. 1. 1.]

t = 1
M usado = 20
[[0.206151 0.203902 0.39871  0.191236]
 [0.208284 0.205341 0.397899 0.188474]
 [0.196758 0.198379 0.400959 0.203902]
 [0.192046 0.193997 0.401471 0.212484]]
Suma de renglones: [0.999999 0.999999 0.999999 0.999999]

t = 5
M usado = 58
[[0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]]
Suma de renglones: [0.999998 0.999998 0.999998 0.999998]


In [7]:
# Tablas de las matrices obtenidas en el ejercicio 3

for t in tiempos:
    P_t = resultados_M[t]
    print(f"\nP({t})")
    display(tabla_matriz(P_t))



P(0.5)


,estado 1,estado 2,estado 3,estado 4
estado 1,0.250609,0.216965,0.386657,0.145770
estado 2,0.253135,0.238361,0.374409,0.134095
estado 3,0.169119,0.193615,0.420301,0.216965
estado 4,0.158017,0.157445,0.398332,0.286206



P(1)


,estado 1,estado 2,estado 3,estado 4
estado 1,0.206151,0.203902,0.398710,0.191236
estado 2,0.208284,0.205341,0.397899,0.188474
estado 3,0.196758,0.198379,0.400959,0.203902
estado 4,0.192046,0.193997,0.401471,0.212484



P(5)


,estado 1,estado 2,estado 3,estado 4
estado 1,0.2,0.2,0.399999,0.2
estado 2,0.2,0.2,0.399999,0.2
estado 3,0.2,0.2,0.399999,0.2
estado 4,0.2,0.2,0.399999,0.2


In [8]:

# Verificación de Chapman-Kolmogorov:
# P(1) debe ser aproximadamente igual a P(0.5)P(0.5)

P_05 = resultados_M[0.5]
P_1 = resultados_M[1]
producto = P_05 @ P_05
diferencia = P_1 - producto

print("P(1):")
print(P_1)

print("\nP(0.5)P(0.5):")
print(producto)

print("\nDiferencia P(1) - P(0.5)P(0.5):")
print(diferencia)

print("\nError máximo absoluto:")
print(np.max(np.abs(diferencia)))


P(1):
[[0.206151 0.203902 0.39871  0.191236]
 [0.208284 0.205341 0.397899 0.188474]
 [0.196758 0.198379 0.400959 0.203902]
 [0.192046 0.193997 0.401471 0.212484]]

P(0.5)P(0.5):
[[0.206151 0.203902 0.39871  0.191236]
 [0.208285 0.205341 0.3979   0.188475]
 [0.196759 0.19838  0.400959 0.203902]
 [0.192047 0.193997 0.401472 0.212485]]

Diferencia P(1) - P(0.5)P(0.5):
[[-0.       -0.       -0.000001 -0.      ]
 [-0.       -0.       -0.000001 -0.      ]
 [-0.       -0.       -0.000001 -0.      ]
 [-0.       -0.       -0.000001 -0.      ]]

Error máximo absoluto:
5.820333638384412e-07



Como el error máximo absoluto es muy pequeño, la ecuación de Chapman-Kolmogorov se verifica numéricamente:

$$P(1) \approx P(0.5)P(0.5).$$


In [9]:
# Ejercicio 4.2: algoritmo con tolerancia epsilon = 0.00001

eps = 0.00001
resultados_eps = {}

for t in tiempos:
    P_t, M, cola = uniformizacion_con_tolerancia(R, t, eps=eps)
    resultados_eps[t] = P_t

    print(f"\nt = {t}")
    print(f"M encontrado = {M}")
    print(f"Cota de cola de Poisson = {cola}")
    print(P_t)
    print("Suma de renglones:", P_t.sum(axis=1))



t = 0.5
M encontrado = 13
Cota de cola de Poisson = 3.4019146132324707e-06
[[0.250608 0.216964 0.386656 0.145769]
 [0.253134 0.23836  0.374408 0.134094]
 [0.169119 0.193614 0.4203   0.216964]
 [0.158017 0.157444 0.39833  0.286205]]
Suma de renglones: [0.999997 0.999997 0.999997 0.999997]

t = 1
M encontrado = 19
Cota de cola de Poisson = 5.180168936913532e-06
[[0.20615  0.203901 0.398708 0.191235]
 [0.208283 0.20534  0.397898 0.188474]
 [0.196758 0.198379 0.400957 0.203901]
 [0.192045 0.193996 0.401469 0.212483]]
Suma de renglones: [0.999995 0.999995 0.999995 0.999995]

t = 5
M encontrado = 56
Cota de cola de Poisson = 7.378955002357301e-06
[[0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]]
Suma de renglones: [0.999993 0.999993 0.999993 0.999993]


In [10]:
# Comparación entre el método con M aproximado y el método con tolerancia

for t in tiempos:
    P_M = resultados_M[t]
    P_eps = resultados_eps[t]
    diferencia = P_M - P_eps

    print(f"\nt = {t}")
    print("Máxima diferencia absoluta entre ambos métodos:")
    print(np.max(np.abs(diferencia)))



t = 0.5
Máxima diferencia absoluta entre ambos métodos:
1.3607613894017767e-06

t = 1
Máxima diferencia absoluta entre ambos métodos:
1.490024779948751e-06

t = 5
Máxima diferencia absoluta entre ambos métodos:
2.200128962071002e-06



## Conclusión

Los dos métodos dan matrices muy parecidas.  
La diferencia principal es que el primer método usa una regla práctica para elegir $M$, mientras que el segundo elige $M$ controlando explícitamente la cola de la distribución Poisson para garantizar una tolerancia $\varepsilon$.
